In [1]:
import pandas as pd
import sys
sys.path.append("../src")
from models import Ride, ride_from_row, ride_serializer

df = pd.read_csv("../src/data/green_tripdata_2019-10.csv")

df = df[["PULocationID", "DOLocationID", "trip_distance", "total_amount", "lpep_pickup_datetime"]]
df.head()

C:\Users\Luca\AppData\Local\Temp\ipykernel_19708\855700584.py:6: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../src/data/green_tripdata_2019-10.csv")


,PULocationID,DOLocationID,trip_distance,total_amount,lpep_pickup_datetime
0,112,196,5.88,19.30,2019-10-01 00:26:02
1,43,263,0.80,9.05,2019-10-01 00:18:11
2,255,228,7.50,22.80,2019-10-01 00:09:31
3,181,181,0.90,6.80,2019-10-01 00:37:40
4,97,188,2.52,13.56,2019-10-01 00:08:13


In [2]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="postgres",
    user="postgres",
    password="postgres"
)

cur = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS rides (
        pickup_location_id INT,
        dropoff_location_id INT,
        trip_distance FLOAT,
        total_amount FLOAT,
        pickup_datetime TIMESTAMP
    );
""")

conn.commit()
print("Tabella creata!")

Tabella creata!


In [3]:
# CREATING KAFKA PRODUCER
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers="localhost:19092",
    value_serializer=ride_serializer
)

print("Producer creato!")

Producer creato!


In [4]:
import time

TOPIC = "rides"

for i, row in df.head(1000).iterrows():
    ride = ride_from_row(row)
    producer.send(TOPIC, value=ride)
    time.sleep(0.01)

producer.flush()
print("Dati inviati!")

Dati inviati!


In [ ]:
# CREATING KAFKA CONSUMER

import sys
sys.path.append("../src")

from kafka import KafkaConsumer
from models import Ride, ride_deserializer
import psycopg2

# connessione postgres
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="postgres",
    user="postgres",
    password="postgres"
)
cur = conn.cursor()

# connessione kafka
consumer = KafkaConsumer(
    "rides",
    bootstrap_servers="localhost:19092",
    value_deserializer=ride_deserializer,
    auto_offset_reset="earliest",
    group_id="rides-consumer"
)

print("Consumer pronto!")